<a href="https://colab.research.google.com/github/alessarana90-hue/NewCode/blob/main/mag(0_00010)MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
!pip install tensorflow

In [ ]:
# Step 1: Import important libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Dropout
from tensorflow.keras.metrics import MeanSquaredError, MeanAbsoluteError
from tensorflow.keras.layers import Dense, Dropout, LSTM, Bidirectional, Input, BatchNormalization
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import math

In [ ]:
# Step 2: Upload the dataset
df = pd.read_csv('/content/drive/MyDrive/The Pacific Ring of Fire.csv')

In [ ]:
df .head(5)

,id,time,latitude,longitude,depth,magnitude,place,tsunami,alert,felt,...,status,sig,net,nst,dmin,gap,magType,type,title,location
0,us6000pfs7,12/26/2024 21:02,30.4847,141.8463,10.000,5.7,"Izu Islands, Japan region",0,green,NaN,...,reviewed,500,us,103.0,3.142,92.0,mww,earthquake,"M 5.7 - Izu Islands, Japan region","Izu Islands, Japan region"
1,us7000nzg7,12/17/2024 4:09,31.0904,130.2373,165.486,5.2,"20 km SSW of Makurazaki, Japan",0,NaN,1.0,...,reviewed,416,us,88.0,0.456,82.0,mb,earthquake,"M 5.2 - 20 km SSW of Makurazaki, Japan","20 km SSW of Makurazaki, Japan"
2,us7000ny8l,12/12/2024 19:55,30.5632,141.9457,10.000,5.1,"Izu Islands, Japan region",0,NaN,NaN,...,reviewed,400,us,87.0,3.125,119.0,mb,earthquake,"M 5.1 - Izu Islands, Japan region","Izu Islands, Japan region"
3,us7000nvxl,12/4/2024 16:53,24.8543,128.0555,10.000,5.0,"146 km SSE of Itoman, Japan",0,NaN,NaN,...,reviewed,385,us,120.0,1.983,103.0,mb,earthquake,"M 5.0 - 146 km SSE of Itoman, Japan","146 km SSE of Itoman, Japan"
4,us7000nv3a,11/30/2024 8:46,25.3021,126.0218,49.015,5.7,"91 km NE of Hirara, Japan",0,green,3.0,...,reviewed,501,us,257.0,1.866,62.0,mww,earthquake,"M 5.7 - 91 km NE of Hirara, Japan","91 km NE of Hirara, Japan"


In [ ]:
df.shape

(27759, 23)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27759 entries, 0 to 27758
Data columns (total 23 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         27759 non-null  object 
 1   time       27759 non-null  object 
 2   latitude   27759 non-null  float64
 3   longitude  27759 non-null  float64
 4   depth      27759 non-null  float64
 5   magnitude  27759 non-null  float64
 6   place      27759 non-null  object 
 7   tsunami    27759 non-null  int64  
 8   alert      1480 non-null   object 
 9   felt       3221 non-null   float64
 10  cdi        3221 non-null   float64
 11  mmi        5460 non-null   float64
 12  country    27759 non-null  object 
 13  status     27759 non-null  object 
 14  sig        27759 non-null  int64  
 15  net        27759 non-null  object 
 16  nst        7991 non-null   float64
 17  dmin       4468 non-null   float64
 18  gap        10205 non-null  float64
 19  magType    27759 non-null  object 
 20  type  

In [ ]:
print("Missing values in each feature:")
print(df.isnull().sum())

Missing values in each feature:
id               0
time             0
latitude         0
longitude        0
depth            0
magnitude        0
place            0
tsunami          0
alert        26279
felt         24538
cdi          24538
mmi          22299
country          0
status           0
sig              0
net              0
nst          19768
dmin         23291
gap          17554
magType          0
type             0
title            0
location         0
dtype: int64


In [ ]:
df.drop(['depth'], axis=1, inplace=True)
#df.drop(['place'], axis=1, inplace=True)
df.drop(['tsunami'], axis=1, inplace=True)
#df.drop(['felt'], axis=1, inplace=True)
#df.drop(['cdi'], axis=1, inplace=True)
df.drop(['mmi'], axis=1, inplace=True)
df.drop(['country'], axis=1, inplace=True)
df.drop(['type'], axis=1, inplace=True)
df.drop(['title'], axis=1, inplace=True)
df.drop(['location'], axis=1, inplace=True)
df.drop(['alert'], axis=1, inplace=True)

In [ ]:
df.head(5)

,id,time,latitude,longitude,magnitude,place,felt,cdi,status,sig,net,nst,dmin,gap,magType
0,us6000pfs7,12/26/2024 21:02,30.4847,141.8463,5.7,"Izu Islands, Japan region",NaN,NaN,reviewed,500,us,103.0,3.142,92.0,mww
1,us7000nzg7,12/17/2024 4:09,31.0904,130.2373,5.2,"20 km SSW of Makurazaki, Japan",1.0,2.2,reviewed,416,us,88.0,0.456,82.0,mb
2,us7000ny8l,12/12/2024 19:55,30.5632,141.9457,5.1,"Izu Islands, Japan region",NaN,NaN,reviewed,400,us,87.0,3.125,119.0,mb
3,us7000nvxl,12/4/2024 16:53,24.8543,128.0555,5.0,"146 km SSE of Itoman, Japan",NaN,NaN,reviewed,385,us,120.0,1.983,103.0,mb
4,us7000nv3a,11/30/2024 8:46,25.3021,126.0218,5.7,"91 km NE of Hirara, Japan",3.0,3.8,reviewed,501,us,257.0,1.866,62.0,mww


In [ ]:
df.drop(['id'], axis=1, inplace=True)
#df.drop(['alert'], axis=1, inplace=True)
#df.drop(['mmi'], axis=1, inplace=True)
df.drop(['status'], axis=1, inplace=True)
df.drop(['sig'], axis=1, inplace=True)
df.drop(['net'], axis=1, inplace=True)
df.drop(['nst'], axis=1, inplace=True)
df.drop(['dmin'], axis=1, inplace=True)
df.drop(['gap'], axis=1, inplace=True)
df.drop(['magType'], axis=1, inplace=True)

In [ ]:
df.head(5)

,time,latitude,longitude,magnitude,place,felt,cdi
0,12/26/2024 21:02,30.4847,141.8463,5.7,"Izu Islands, Japan region",NaN,NaN
1,12/17/2024 4:09,31.0904,130.2373,5.2,"20 km SSW of Makurazaki, Japan",1.0,2.2
2,12/12/2024 19:55,30.5632,141.9457,5.1,"Izu Islands, Japan region",NaN,NaN
3,12/4/2024 16:53,24.8543,128.0555,5.0,"146 km SSE of Itoman, Japan",NaN,NaN
4,11/30/2024 8:46,25.3021,126.0218,5.7,"91 km NE of Hirara, Japan",3.0,3.8


In [ ]:
#df.drop(['type'], axis=1, inplace=True)
#df.drop(['title'], axis=1, inplace=True)
#df.drop(['location'], axis=1, inplace=True)

In [ ]:
df.drop(['latitude'], axis=1, inplace=True)
df.drop(['longitude'], axis=1, inplace=True)
#df.drop(['depth'], axis=1, inplace=True)
#df.drop(['country'], axis=1, inplace=True)

In [ ]:
df.head(5)

,time,magnitude,place,felt,cdi
0,12/26/2024 21:02,5.7,"Izu Islands, Japan region",NaN,NaN
1,12/17/2024 4:09,5.2,"20 km SSW of Makurazaki, Japan",1.0,2.2
2,12/12/2024 19:55,5.1,"Izu Islands, Japan region",NaN,NaN
3,12/4/2024 16:53,5.0,"146 km SSE of Itoman, Japan",NaN,NaN
4,11/30/2024 8:46,5.7,"91 km NE of Hirara, Japan",3.0,3.8


In [ ]:
#df = pd.DataFrame(df)

# تحويل عمود الوقت إلى نوع تاريخي
#df['time'] = pd.to_datetime(df['time'])

# الاحتفاظ فقط بالتاريخ
#df['time'] = df['time'].dt.date

In [ ]:
df.head(5)

,time,magnitude,place,felt,cdi
0,12/26/2024 21:02,5.7,"Izu Islands, Japan region",NaN,NaN
1,12/17/2024 4:09,5.2,"20 km SSW of Makurazaki, Japan",1.0,2.2
2,12/12/2024 19:55,5.1,"Izu Islands, Japan region",NaN,NaN
3,12/4/2024 16:53,5.0,"146 km SSE of Itoman, Japan",NaN,NaN
4,11/30/2024 8:46,5.7,"91 km NE of Hirara, Japan",3.0,3.8


In [ ]:
df.shape

(27759, 5)

In [ ]:
# Handle remaining missing values
# Fill numeric columns with the mean
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

In [ ]:
# Fill categorical columns with the mode (most frequent value)
categorical_cols = df.select_dtypes(include=[object]).columns
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Detect and manage outliers using IQR
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df

# Apply outlier removal to numerical columns
for col in numeric_cols:
    df = remove_outliers(df, col)

<ipython-input-20-969f4d23c9fb>:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
df.shape

(23416, 5)

In [ ]:
# Handle categorical data (assuming 'description' is a column with string values)
if 'description' in df.columns:
    df.drop(['description'], axis=1, inplace=True)

In [ ]:
df.head(5)

,time,magnitude,place,felt,cdi
0,12/26/2024 21:02,5.7,"Izu Islands, Japan region",33.967401,3.782614
2,12/12/2024 19:55,5.1,"Izu Islands, Japan region",33.967401,3.782614
3,12/4/2024 16:53,5.0,"146 km SSE of Itoman, Japan",33.967401,3.782614
7,11/25/2024 15:37,5.0,"Izu Islands, Japan region",33.967401,3.782614
12,11/9/2024 5:00,5.0,"Izu Islands, Japan region",33.967401,3.782614


In [ ]:
# Ensure all features are numeric
for column in df.columns:
    if df[column].dtype == 'object':
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])

In [ ]:
df.head(5)

,time,magnitude,place,felt,cdi
0,7141,5.7,17248,33.967401,3.782614
2,6138,5.1,17248,33.967401,3.782614
3,7640,5.0,4017,33.967401,3.782614
7,5110,5.0,17248,33.967401,3.782614
12,5855,5.0,17248,33.967401,3.782614


In [ ]:
# Step 3: Do preprocessing
# Extract features and target
X = df.drop(columns=['magnitude'])
y = df['magnitude']

In [ ]:
X.shape

(23416, 4)

In [ ]:
X.head(5)

,time,place,felt,cdi
0,7141,17248,33.967401,3.782614
2,6138,17248,33.967401,3.782614
3,7640,4017,33.967401,3.782614
7,5110,17248,33.967401,3.782614
12,5855,17248,33.967401,3.782614


In [ ]:
# Fit a Random Forest to determine feature importances
#rf = RandomForestRegressor()
#rf.fit(X, y)

# Get feature importances and sort them
#importances = rf.feature_importances_
#indices = np.argsort(importances)[::-1]
#selected_features = X.columns[indices[:15]]  # Select top 20 features

# Update X to include only the selected features
#X_selected = X[selected_features]

In [ ]:
#X_selected.head(5)

In [ ]:
# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Step 1: Split data into 80% training+validation and 20% testing
X_text_train_val, X_text_test, y_train_val, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Step 2: Split the 80% training+validation set into 70% training and 10% validation
X_text_train, X_text_val, y_train, y_val = train_test_split(X_text_train_val, y_train_val, test_size=0.125, random_state=42)

# Check the shapes to ensure the correct split
print(f"Training set size: {X_text_train.shape[0]} samples")
print(f"Validation set size: {X_text_val.shape[0]} samples")
print(f"Testing set size: {X_text_test.shape[0]} samples")

Training set size: 16390 samples
Validation set size: 2342 samples
Testing set size: 4684 samples


In [ ]:
# Import necessary libraries for each model
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error

In [ ]:
# Helper function to train and evaluate models
def evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test):
    # Fit the model
    model.fit(X_train, y_train)

    # Evaluate on validation data
    y_val_pred = model.predict(X_val)
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_rmse = math.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # Evaluate on test data
    y_test_pred = model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = math.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"Validation MSE: {val_mse}, MAE: {val_mae}, RMSE: {val_rmse}, R²: {val_r2}")
    print(f"Test MSE: {test_mse}, MAE: {test_mae}, RMSE: {test_rmse}, R²: {test_r2}")
    print("="*60)

    return model

In [ ]:
# 1. Support Vector Machine (SVM)
print("Support Vector Machine:")
svm = SVR(kernel='linear')  # Use 'linear' kernel for regression tasks
evaluate_model(svm, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Support Vector Machine:
Validation MSE: 0.13964595221071352, MAE: 0.26662297925550443, RMSE: 0.37369232292183036, R²: -0.1791131543450255
Test MSE: 0.13784686089224815, MAE: 0.2663828680818852, RMSE: 0.37127733689554515, R²: -0.18680646979063287


SVR(kernel='linear')

In [ ]:
# 1. Support Vector Machine (SVM)
print("Support Vector Machine:")
svm = SVR(kernel='rbf')  # Use 'linear' kernel for regression tasks
evaluate_model(svm, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Support Vector Machine:
Validation MSE: 0.13025182057777318, MAE: 0.2672003694438896, RMSE: 0.3609041709065901, R²: -0.0997929591893878
Test MSE: 0.12842725977075553, MAE: 0.266028192208698, RMSE: 0.35836749262559453, R²: -0.1057074626643617


SVR()

In [ ]:
# Predict values after evaluation
y_val_pred = svm.predict(X_text_val)
y_test_pred = svm.predict(X_text_test)

# Print predictions
print("Predicted values on Validation Set:")
print(y_val_pred)

print("\nPredicted values on Testing Set:")
print(y_test_pred)

Predicted values on Validation Set:
[5.20781997 5.22766561 5.19670926 ... 5.21516775 5.19986234 5.28222148]

Predicted values on Testing Set:
[5.25093884 5.20241491 5.25055011 ... 5.22163161 5.24879111 5.25905871]


In [ ]:
# Create DataFrames for better readability
val_comparison = pd.DataFrame({'True Value': y_val, 'Predicted Value': y_val_pred})
test_comparison = pd.DataFrame({'True Value': y_test, 'Predicted Value': y_test_pred})

print("Validation Set: True vs Predicted")
print(val_comparison.head(10))  # Show first 10 rows

print("\nTesting Set: True vs Predicted")
print(test_comparison.head(10))  # Show first 10 rows

Validation Set: True vs Predicted
       True Value  Predicted Value
23924        5.30         5.207820
23672        5.00         5.227666
7247         5.00         5.196709
27607        5.88         5.294041
12197        5.20         5.199969
12254        5.30         5.269212
26148        5.10         5.296312
15765        6.00         5.237259
3725         5.40         5.204357
21508        5.50         5.216330

Testing Set: True vs Predicted
       True Value  Predicted Value
5114         6.00         5.250939
18876        5.20         5.202415
12011        5.00         5.250550
25594        5.90         5.253106
18103        5.60         5.203451
6672         5.00         5.227963
21085        5.98         5.236992
13262        5.40         5.248949
6602         5.30         5.269090
11288        5.00         5.229801


In [ ]:
# 1. Support Vector Machine (SVM)
print("Support Vector Machine:")
svm = SVR(kernel='poly')  # Use 'linear' kernel for regression tasks
evaluate_model(svm, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Support Vector Machine:
Validation MSE: 0.139633195867787, MAE: 0.26662264178369066, RMSE: 0.37367525455639555, R²: -0.17900544501648508
Test MSE: 0.1378362902080929, MAE: 0.2663749230023789, RMSE: 0.37126310105919885, R²: -0.18671546041788134


SVR(kernel='poly')

In [ ]:
# 2. Random Forest Regressor
print("Random Forest Regressor:")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
evaluate_model(rf, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Random Forest Regressor:
Validation MSE: 0.1292822059314214, MAE: 0.285394200113863, RMSE: 0.3595583484379432, R²: -0.09160593073593248
Test MSE: 0.12483423100199256, MAE: 0.2807964560204954, RMSE: 0.35331888005312223, R²: -0.07477291862534297


RandomForestRegressor(random_state=42)

In [ ]:
#y_test.to_csv('y_test.csv', index=False)

In [ ]:
# Print real vs predicted values for test set
y_test_pred = rf.predict(X_text_test)

print("\nTest Set - Real vs Predicted:")
for real, predicted in zip(y_test, y_test_pred):
    print(f"Real: {real}, Predicted: {predicted}")


Test Set - Real vs Predicted:
Real: 6.0, Predicted: 5.3923000000000005
Real: 5.2, Predicted: 5.355900000000002
Real: 5.0, Predicted: 5.137100000000003
Real: 5.9, Predicted: 5.280299999999999
Real: 5.6, Predicted: 5.2120000000000015
Real: 5.0, Predicted: 5.377600000000006
Real: 5.98, Predicted: 5.249000000000003
Real: 5.4, Predicted: 5.301600000000004
Real: 5.3, Predicted: 5.441800000000003
Real: 5.0, Predicted: 5.375799999999995
Real: 5.2, Predicted: 5.324599999999999
Real: 6.1, Predicted: 5.454999999999997
Real: 5.8, Predicted: 5.307699999999996
Real: 5.0, Predicted: 5.232000000000004
Real: 5.0, Predicted: 5.291800000000003
Real: 5.0, Predicted: 5.246100000000002
Real: 5.73, Predicted: 5.7103
Real: 5.0, Predicted: 5.293500000000003
Real: 5.3, Predicted: 5.169000000000004
Real: 6.1, Predicted: 5.336799999999999
Real: 5.3, Predicted: 5.5147999999999975
Real: 5.1, Predicted: 5.262799999999999
Real: 5.1, Predicted: 5.3820000000000014
Real: 5.2, Predicted: 5.284999999999997
Real: 5.64, Pr

In [ ]:
# 3. Decision Tree Regressor
print("Decision Tree Regressor:")
dt = DecisionTreeRegressor(random_state=42)
evaluate_model(dt, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Decision Tree Regressor:
Validation MSE: 0.22132963279248508, MAE: 0.35297181895815544, RMSE: 0.47045683414367045, R²: -0.8688166562693374
Test MSE: 0.2103037788215201, MAE: 0.34567250213492745, RMSE: 0.4585888995838431, R²: -0.8106316220134839


DecisionTreeRegressor(random_state=42)

In [ ]:
# 4. K-Nearest Neighbors (KNN) Regressor
print("K-Nearest Neighbors Regressor:")
knn = KNeighborsRegressor(n_neighbors=5)  # You can tune 'n_neighbors' for better results
evaluate_model(knn, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

K-Nearest Neighbors Regressor:
Validation MSE: 0.1410865038428694, MAE: 0.2988266438941076, RMSE: 0.3756148344286596, R²: -0.19127658158439997
Test MSE: 0.1359815345858241, MAE: 0.29275875320239114, RMSE: 0.36875674174965817, R²: -0.17074675457908972


KNeighborsRegressor()

In [ ]:
# 5. Logistic Regression
print("Logistic Regression:")
lr = LinearRegression()  # Increased iterations for convergence
evaluate_model(lr, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Logistic Regression:
Validation MSE: 0.11847802677016955, MAE: 0.28057968789515203, RMSE: 0.34420637235555296, R²: -0.0003798724846348911
Test MSE: 0.11601474984079387, MAE: 0.277784162374302, RMSE: 0.3406093801421122, R²: 0.0011592950976018201


LinearRegression()